In [8]:
import nbformat
filepath="MODULE_A.ipynb"
# Open Module A
with open(filepath, "r", encoding="utf-8") as file:
    module_A = nbformat.read(file, as_version=4)

print("Module A loaded successfully!")

FileNotFoundError: [Errno 2] No such file or directory: 'MODULE_A.ipynb'

In [ ]:
# Run the code from Module A

for cell in module_A.cells:
    if cell.cell_type == "code":
        exec(cell.source)

print("Module A executed successfully!")

In [ ]:
#test 1
import pandas as pd

# Load the original input dataset
input_data = pd.read_csv("isro_burnin_ess_synthetic_dataset_cleaned.csv")

# Load Module A's output
output_data = pd.read_csv("module_a_output.csv")

# Count the number of components
number_of_components = input_data["component_id"].nunique()

# Module A should create 3 rows for every component
expected_rows = number_of_components * 3

# Count the rows actually produced
actual_rows = len(output_data)

# Check the result
if actual_rows == expected_rows:
    print("TEST 1 PASSED ")
    print("Expected rows:", expected_rows)
    print("Actual rows:", actual_rows)
else:
    print("TEST 1 FAILED ")
    print("Expected rows:", expected_rows)
    print("Actual rows:", actual_rows)

In [ ]:
#test 2: checking for all necessary columns
# Required columns from Module A
required_columns = [
    "lot_id",
    "part_id",
    "param_name",
    "value_0h",
    "value_24h",
    "value_96h",
    "value_168h",
    "drift_0_24",
    "drift_24_96",
    "drift_96_168",
    "total_drift_pct",
    "value_0h_median",
    "value_0h_sigma_robust",
    "value_0h_UPL",
    "value_0h_LPL",
    "value_0h_DPAT_outlier",
    "total_drift_pct_median",
    "total_drift_pct_sigma_robust",
    "total_drift_pct_UPL",
    "total_drift_pct_LPL",
    "total_drift_pct_DPAT_outlier",
    "iso_forest_flag",
    "iso_forest_score",
    "final_flag",
    "flag_reason"
]

# Find missing columns
missing_columns = []

for column in required_columns:
    if column not in output_data.columns:
        missing_columns.append(column)

# Check the result
if len(missing_columns) == 0:
    print("TEST 2 PASSED ")
    print("All required columns are present.")
else:
    print("TEST 2 FAILED ")
    print("Missing columns:", missing_columns)

In [ ]:
#checking for the presence of all 3 parameters
# The three parameters expected for every part
expected_parameters = {"iddq", "leakage", "prop_delay"}

# Group the output by part
part_groups = output_data.groupby("part_id")

# Store any parts that have a problem
problem_parts = []

# Check every part
for part_id, group in part_groups:

    # Get the parameters for this part
    parameters = set(group["param_name"])

    # Check if the parameters are exactly the expected three
    if parameters != expected_parameters:
        problem_parts.append(part_id)

# Final result
if len(problem_parts) == 0:
    print("TEST 3 PASSED ")
    print("Every part has exactly the three required parameters.")
else:
    print("TEST 3 FAILED ")
    print("Number of problematic parts:", len(problem_parts))
    print("Example problematic parts:", problem_parts[:10])

In [ ]:
# Calculate drift ourselves
expected_drift = (
    output_data["value_24h"] - output_data["value_0h"]
) / 24

# Compare with Module A's drift
difference = abs(
    output_data["drift_0_24"] - expected_drift
)

# Allow a very tiny difference because of decimal rounding
tolerance = 0.000001

# Check every row
if (difference <= tolerance).all():
    print("TEST 4 PASSED ")
    print("drift_0_24 is calculated correctly.")
else:
    print("TEST 4 FAILED ")

    # Show how many rows are incorrect
    wrong_rows = (difference > tolerance).sum()
    print("Incorrect rows:", wrong_rows)

In [ ]:
# Calculate the expected drift ourselves
expected_drift = (
    output_data["value_96h"] - output_data["value_24h"]
) / 72

# Compare with Module A's result
difference = abs(
    output_data["drift_24_96"] - expected_drift
)

# Allow a tiny difference because of decimal calculations
tolerance = 0.000001

# Check every row
if (difference <= tolerance).all():
    print("TEST 5 PASSED ")
    print("drift_24_96 is calculated correctly.")
else:
    print("TEST 5 FAILED ")
    print("Incorrect rows:", (difference > tolerance).sum())

In [ ]:
# Calculate the expected drift ourselves
expected_drift = (
    output_data["value_168h"] - output_data["value_96h"]
) / 72

# Compare with Module A's result
difference = abs(
    output_data["drift_96_168"] - expected_drift
)

# Allow a tiny difference because of decimal calculations
tolerance = 0.000001

# Check every row
if (difference <= tolerance).all():
    print("TEST 6 PASSED ")
    print("drift_96_168 is calculated correctly.")
else:
    print("TEST 6 FAILED ")
    print("Incorrect rows:", (difference > tolerance).sum())

In [ ]:
# Calculate the expected total drift
expected_drift = (
    (output_data["value_168h"] - output_data["value_0h"])
    / output_data["value_0h"]
)

# Handle value_0h = 0 exactly like Module A
expected_drift = expected_drift.where(
    output_data["value_0h"] != 0,
    0.0
)

# Compare with Module A's result
difference = abs(
    output_data["total_drift_pct"] - expected_drift
)

# Allow a tiny difference because of decimal calculations
tolerance = 0.000001

# Check every row
if (difference <= tolerance).all():
    print("TEST 7 PASSED ")
    print("total_drift_pct is calculated correctly.")
else:
    print("TEST 7 FAILED ")
    print("Incorrect rows:", (difference > tolerance).sum())

In [ ]:
'''# Check DPAT limits for every lot and parameter group

tolerance = 0.000001
failed_groups = []

for (lot_id, param_name), group in output_data.groupby(
    ["lot_id", "param_name"]
):

    # -------------------------
    # Calculate expected values
    # -------------------------

    values = group["value_0h"].values

    median = np.median(values)

    mad = np.median(np.abs(values - median))

    sigma = 1.4826 * mad

    upl = median + 6 * sigma

    lpl = median - 6 * sigma

    # -------------------------
    # Compare with Module A
    # -------------------------

    if not np.allclose(
        group["value_0h_median"], median, atol=tolerance
    ):
        failed_groups.append((lot_id, param_name, "median"))

    if not np.allclose(
        group["value_0h_sigma_robust"], sigma, atol=tolerance
    ):
        failed_groups.append((lot_id, param_name, "sigma"))

    if not np.allclose(
        group["value_0h_UPL"], upl, atol=tolerance
    ):
        failed_groups.append((lot_id, param_name, "UPL"))

    if not np.allclose(
        group["value_0h_LPL"], lpl, atol=tolerance
    ):
        failed_groups.append((lot_id, param_name, "LPL"))


# -------------------------
# Final result
# -------------------------

if len(failed_groups) == 0:
    print("TEST 8 PASSED ")
    print("All value_0h DPAT limits are correct.")
else:
    print("TEST 8 FAILED ")
    print("Number of failed checks:", len(failed_groups))
    print("Examples:", failed_groups[:10])'''

In [ ]:
'''# Calculate the expected 0h DPAT outlier decision

expected_outlier = (
    (output_data["value_0h"] > output_data["value_0h_UPL"])
    |
    (output_data["value_0h"] < output_data["value_0h_LPL"])
)

# Compare with Module A's result
if (
    expected_outlier.astype(bool)
    == output_data["value_0h_DPAT_outlier"].astype(bool)
).all():

    print("TEST 9 PASSED ")
    print("value_0h DPAT outlier detection is correct.")

else:

    print("TEST 9 FAILED ")

    wrong_rows = (
        expected_outlier.astype(bool)
        != output_data["value_0h_DPAT_outlier"].astype(bool)
    ).sum()

    print("Incorrect rows:", wrong_rows)'''

In [ ]:
# Check DPAT limits for total drift

tolerance = 0.000001
failed_groups = []

for (lot_id, param_name), group in output_data.groupby(
    ["lot_id", "param_name"]
):

    # Get total drift values
    values = group["total_drift_pct"].values

    # Calculate expected DPAT values
    median = np.median(values)

    mad = np.median(np.abs(values - median))

    sigma = 1.4826 * mad

    upl = median + 6 * sigma

    lpl = median - 6 * sigma

    # Compare with Module A
    if not np.allclose(
        group["total_drift_pct_median"], median, atol=tolerance
    ):
        failed_groups.append((lot_id, param_name, "median"))

    if not np.allclose(
        group["total_drift_pct_sigma_robust"], sigma, atol=tolerance
    ):
        failed_groups.append((lot_id, param_name, "sigma"))

    if not np.allclose(
        group["total_drift_pct_UPL"], upl, atol=tolerance
    ):
        failed_groups.append((lot_id, param_name, "UPL"))

    if not np.allclose(
        group["total_drift_pct_LPL"], lpl, atol=tolerance
    ):
        failed_groups.append((lot_id, param_name, "LPL"))


# Final result
if len(failed_groups) == 0:
    print("TEST 10 PASSED ")
    print("All total drift DPAT limits are correct.")
else:
    print("TEST 10 FAILED ")
    print("Number of failed checks:", len(failed_groups))
    print("Examples:", failed_groups[:10])

In [ ]:
'''# Calculate the expected drift outlier decision

expected_outlier = (
    (output_data["total_drift_pct"] > output_data["total_drift_pct_UPL"])
    |
    (output_data["total_drift_pct"] < output_data["total_drift_pct_LPL"])
)

# Compare with Module A's result
actual_outlier = output_data["total_drift_pct_DPAT_outlier"].astype(bool)

# Check every row
if (expected_outlier == actual_outlier).all():

    print("TEST 11 PASSED ")
    print("Total-drift DPAT outlier detection is correct.")

else:

    print("TEST 11 FAILED ")

    wrong_rows = (expected_outlier != actual_outlier).sum()

    print("Incorrect rows:", wrong_rows)'''

In [ ]:
'''# Check that Isolation Forest flags contain only True or False

valid_flags = {True, False}

actual_flags = set(
    output_data["iso_forest_flag"].dropna().unique()
)

if actual_flags.issubset(valid_flags):
    print("TEST 12 PASSED ")
    print("Isolation Forest flags contain only True/False values.")
else:
    print("TEST 12 FAILED ")
    print("Unexpected values found:", actual_flags)'''

In [ ]:
# Check Isolation Forest scores

scores = output_data["iso_forest_score"]

if (
    scores.notna().all()
    and
    np.isfinite(scores).all()
    and
    np.issubdtype(scores.dtype, np.number)
):
    print("TEST 13 PASSED ")
    print("All Isolation Forest scores are valid numeric values.")
else:
    print("TEST 13 FAILED ")

    print("Missing values:", scores.isna().sum())
    print("Infinite values:", np.isinf(scores).sum())

In [ ]:
# Calculate the expected final flag ourselves

expected_final = (
    output_data["value_0h_DPAT_outlier"].astype(bool)
    |
    output_data["total_drift_pct_DPAT_outlier"].astype(bool)
    |
    output_data["iso_forest_flag"].astype(bool)
)

# Get Module A's final flag
actual_final = output_data["final_flag"].astype(bool)

# Compare them
if (expected_final == actual_final).all():

    print("TEST 14 PASSED ")
    print("final_flag correctly combines all three detectors.")

else:

    print("TEST 14 FAILED ")

    wrong_rows = (expected_final != actual_final).sum()

    print("Incorrect rows:", wrong_rows)

In [ ]:
# Convert missing flag_reason values back to an empty string
actual_reason = output_data["flag_reason"].fillna("")

# Calculate the expected reason
expected_reason = output_data.apply(
    get_expected_reason,
    axis=1
)

# Compare the two
if (expected_reason == actual_reason).all():

    print("TEST 15 PASSED ")
    print("flag_reason correctly matches the anomaly detectors.")

else:

    print("TEST 15 FAILED ")

    wrong_rows = (expected_reason != actual_reason).sum()

    print("Incorrect rows:", wrong_rows)

In [ ]:
# Find parts that Module A says are rejected
rejected_parts = set(
    output_data.loc[
        output_data["final_flag"],
        "part_id"
    ].unique()
)

# Find all parts in the output
all_parts = set(output_data["part_id"].unique())

# Check every part
wrong_parts = []

for part_id in all_parts:

    # Does this part have at least one flagged evaluation?
    has_flag = output_data.loc[
        output_data["part_id"] == part_id,
        "final_flag"
    ].any()

    # Is the part present in the rejected list?
    is_rejected = part_id in rejected_parts

    # They should always agree
    if has_flag != is_rejected:
        wrong_parts.append(part_id)


# Final result
if len(wrong_parts) == 0:

    print("TEST 16 PASSED ")
    print("Rejected part_ids are consistent with final_flag.")

else:

    print("TEST 16 FAILED ")
    print("Inconsistent parts:", len(wrong_parts))
    print("Examples:", wrong_parts[:10])

In [ ]:
# Columns that should not contain missing or invalid values
important_columns = [
    "lot_id",
    "part_id",
    "param_name",
    "value_0h",
    "value_24h",
    "value_96h",
    "value_168h",
    "drift_0_24",
    "drift_24_96",
    "drift_96_168",
    "total_drift_pct",
    "iso_forest_score",
    "final_flag"
]

problems = []

for column in important_columns:

    # Check for missing values
    missing = output_data[column].isna().sum()

    if missing > 0:
        problems.append(
            f"{column}: {missing} missing values"
        )

    # Check numeric columns for infinity
    if output_data[column].dtype != "object":

        infinite = np.isinf(output_data[column]).sum()

        if infinite > 0:
            problems.append(
                f"{column}: {infinite} infinite values"
            )


# Check Boolean flag columns
flag_columns = [
    "value_0h_DPAT_outlier",
    "total_drift_pct_DPAT_outlier",
    "iso_forest_flag",
    "final_flag"
]

for column in flag_columns:

    valid_values = {True, False}

    actual_values = set(
        output_data[column].dropna().unique()
    )

    if not actual_values.issubset(valid_values):
        problems.append(
            f"{column}: invalid flag values {actual_values}"
        )


# Final result
if len(problems) == 0:

    print("TEST 17 PASSED ")
    print("Output data contains no missing or invalid values in the important fields.")

else:

    print("TEST 17 FAILED ")

    for problem in problems:
        print("-", problem)